# ASMSA v2 — Live metadynamics monitor

← [Metadynamics setup and run](06-md.ipynb) &nbsp;|&nbsp; [Analysis](07-analysis.ipynb) →

Open this notebook in a **separate kernel** while production is running in
[06-md](06-md.ipynb). It watches the growing `COLVAR`, `HILLS`, and GROMACS log
files in the same project `04-md` directory. It does not start, stop, or modify
the simulation.

The plots give a live indication of CV exploration, deposited bias, and elapsed
simulation time. They are diagnostic only—not a convergence test. The monitor
intentionally does not read `md.xtc` while it is being written, since a partial
trajectory may not yet be readable. When the live CVs leave the saved training
latent's coordinate range, it reports a warning; check the PLUMED export and
periodic-boundary handling before trusting such a run.

In [ ]:
from collections import deque
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display

import asmsa_v2 as asmsa
from asmsa_v2.viz import use_asmsa_style

use_asmsa_style()
project = asmsa.Project.load('run')
md_dir = project.dir('md')
print(f'Project root: {project.root.resolve()}')
print(f'Monitoring:   {md_dir.resolve()}')
print(f'Target run:   {project.production_ns:g} ns')

## Start the live view

Run the next cell after the production cell in 06-md has started. It refreshes every
`REFRESH_SECONDS`; interrupt the cell (⏹) to stop monitoring. This only stops the
display loop—the separately launched simulation continues.

The progress percentage is an estimate based on the latest `COLVAR` time divided by
the configured production length. A stale file can mean the run is paused, finished,
or has encountered an issue; inspect the log tail below rather than treating it as a
definitive process status.

In [ ]:
REFRESH_SECONDS = 15
MAX_PLOT_POINTS = 10000
colvar_path = md_dir / 'COLVAR'
hills_path = md_dir / 'HILLS'
log_path = md_dir / 'logs' / 'gmx-mdrun.log'
train_latent_path = project.dir('models') / f'latent-{project.model_name}.npy'
training_latent = None
if train_latent_path.exists():
    training_latent = np.load(train_latent_path)
    if training_latent.ndim != 2 or training_latent.shape[1] < 2:
        raise ValueError(f'Expected at least two latent dimensions in {train_latent_path}; '
                         f'got shape {training_latent.shape}')
    print(f'Training latent reference: {train_latent_path} ({len(training_latent):,} points)')
else:
    print(f'No training latent found at {train_latent_path}; CV-space plot will show MD only.')

def new_reader(max_points):
    return {
        'identity': None, 'offset': 0, 'pending': b'', 'fields': None,
        'rows': deque(maxlen=max_points), 'count': 0,
    }

colvar_reader = new_reader(MAX_PLOT_POINTS)
hills_reader = new_reader(1)

def read_appended_rows(path, state):
    if not path.exists():
        return state

    stat = path.stat()
    identity = (stat.st_dev, stat.st_ino)
    if state['identity'] != identity or stat.st_size < state['offset']:
        state.update(identity=identity, offset=0, pending=b'', fields=None,
                     rows=deque(maxlen=state['rows'].maxlen), count=0)

    with path.open('rb') as stream:
        stream.seek(state['offset'])
        chunk = stream.read()
        state['offset'] += len(chunk)

    content = state['pending'] + chunk
    if b'\n' not in content:
        state['pending'] = content
        return state
    complete, state['pending'] = content.rsplit(b'\n', 1)
    if content.endswith(b'\n'):
        state['pending'] = b''

    for raw_line in complete.splitlines():
        line = raw_line.decode('utf-8', errors='replace').strip()
        if not line:
            continue
        if line.startswith('#! FIELDS'):
            state['fields'] = line.split()[2:]
            continue
        if line.startswith('#') or line.startswith('@'):
            continue
        values = [float(value) for value in line.split()]
        if state['fields'] is not None and len(values) != len(state['fields']):
            raise ValueError(f'Unexpected column count in {path.name}: {line}')
        state['rows'].append(values)
        state['count'] += 1
    return state

def file_status(path):
    if not path.exists():
        return 'not created yet'
    age = max(0, time.time() - path.stat().st_mtime)
    size_mb = path.stat().st_size / (1024 * 1024)
    freshness = (f'updated {age:.0f} s ago' if age < max(120, 4 * REFRESH_SECONDS)
                 else f'stale; last update {age / 60:.1f} min ago')
    return f'{size_mb:.2f} MB; {freshness}'

def log_tail(path, lines=12):
    if not path.exists():
        return 'No GROMACS log found at the expected path yet.'
    with path.open('rb') as stream:
        stream.seek(max(0, path.stat().st_size - 8192))
        text = stream.read().decode('utf-8', errors='replace')
    return '\n'.join(text.splitlines()[-lines:]) or '(log is empty)'

def render_dashboard():
    read_appended_rows(colvar_path, colvar_reader)
    read_appended_rows(hills_path, hills_reader)
    clear_output(wait=True)

    print(f'Live metadynamics monitor — {time.strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'Project: {project.root.resolve()}')
    print(f'COLVAR:  {file_status(colvar_path)}')
    print(f'HILLS:   {file_status(hills_path)}')
    print(f'Trajectory: {file_status(md_dir / "md.xtc")} (size/update only; not opened)')
    print(f'GROMACS log: {log_path}')

    fields = colvar_reader['fields'] or []
    rows = list(colvar_reader['rows'])
    if not rows or not fields:
        print('\nWaiting for complete COLVAR data rows. The simulation may not have started yet.')
    else:
        data = np.asarray(rows, dtype=float)
        columns = {name: data[:, i] for i, name in enumerate(fields)}
        time_col = 'time' if 'time' in columns else fields[0]
        time_ns = columns[time_col] / 1000.0
        latest_ns = float(time_ns[-1])
        pct = 100 * latest_ns / project.production_ns if project.production_ns > 0 else float('nan')
        print(f'\nCOLVAR records read: {colvar_reader["count"]:,} '
              f'(plot shows latest {len(rows):,})')
        print(f'Latest simulation time: {latest_ns:.3f} ns / '
              f'{project.production_ns:g} ns ({pct:.1f}% estimated)')
        if hills_reader['count']:
            print(f'Deposited hills observed: {hills_reader["count"]:,}')

        bias_cols = [name for name in fields if name.endswith('.bias')]
        cv_cols = [name for name in fields if name.startswith('model.node-')]
        if not cv_cols:
            cv_cols = [name for name in fields
                       if name not in {time_col, *bias_cols}
                       and not name.endswith('.rct')]
        cv_cols = cv_cols[:2]
        if (training_latent is not None
                and cv_cols == ['model.node-0', 'model.node-1']):
            train_min = training_latent[:, :2].min(axis=0)
            train_max = training_latent[:, :2].max(axis=0)
            live_min = data[:, [fields.index(name) for name in cv_cols]].min(axis=0)
            live_max = data[:, [fields.index(name) for name in cv_cols]].max(axis=0)
            outside = np.any(live_min < train_min) or np.any(live_max > train_max)
            print(f'Training CV range: [{train_min[0]:.3g}, {train_max[0]:.3g}] x '
                  f'[{train_min[1]:.3g}, {train_max[1]:.3g}]')
            print(f'Live CV range:     [{live_min[0]:.3g}, {live_max[0]:.3g}] x '
                  f'[{live_min[1]:.3g}, {live_max[1]:.3g}]')
            if outside:
                print('WARNING: live CV values are outside the training latent range. '
                      'Check the PLUMED export and PBC handling; this is not just a plot mismatch.')

        panels = 1 + bool(bias_cols) + (1 if len(cv_cols) >= 2 else 0)
        fig, axes = plt.subplots(panels, 1, figsize=(10, 2.6 * panels), squeeze=False)
        ax_index = 0
        ax = axes[ax_index, 0]
        for name in cv_cols:
            ax.plot(time_ns, columns[name], lw=0.7, label=name)
        ax.set_ylabel('CV value')
        ax.set_xlabel('time (ns)')
        if cv_cols:
            ax.legend(loc='best', fontsize=8)
        else:
            ax.text(0.5, 0.5, 'No CV columns found in COLVAR', ha='center', va='center',
                    transform=ax.transAxes)
        ax_index += 1

        if bias_cols:
            ax = axes[ax_index, 0]
            for name in bias_cols:
                ax.plot(time_ns, columns[name], lw=0.7, label=name)
            ax.set_ylabel('bias (kJ/mol)')
            ax.set_xlabel('time (ns)')
            ax.legend(loc='best', fontsize=8)
            ax_index += 1

        if len(cv_cols) >= 2:
            ax = axes[ax_index, 0]
            if (training_latent is not None
                    and cv_cols[:2] == ['model.node-0', 'model.node-1']):
                stride = max(1, len(training_latent) // 20000)
                ax.scatter(training_latent[::stride, 0], training_latent[::stride, 1],
                           s=4, c='0.65', alpha=0.35, label='training latent',
                           zorder=1)
            ax.scatter(columns[cv_cols[0]], columns[cv_cols[1]], s=5,
                       c='crimson', alpha=0.55, label='metadynamics', zorder=2)
            ax.set_xlabel(cv_cols[0])
            ax.set_ylabel(cv_cols[1])
            ax.set_title('CV space explored so far')
            ax.legend(loc='best', fontsize=8, markerscale=2)

        fig.tight_layout()
        display(fig)
        plt.close(fig)

    print('\nGROMACS log tail:')
    print(log_tail(log_path))

try:
    while True:
        render_dashboard()
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print('\nMonitor stopped. The simulation was not interrupted.')